# Final Analysis – Part I

This notebook answers the final part of this project, where we bring the pieces together. Using cross-validation, we perform a final model selection for the Runge function with all three methods: OLS as a function of the polynomial degree, and Ridge and Lasso as functions of both the polynomial degree and $\lambda$. We identify the best model according to the cross-validated test error, and discuss the strenghts and weaknesses of the three models for this problem.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from fys_stk4155_p1.data.design_matrix import univariate_polynomial_design_matrix
from fys_stk4155_p1.data.runge import generate_runge_data, runge_function
from fys_stk4155_p1.metrics import mean_squared_error
from fys_stk4155_p1.regression.lasso import Lasso
from fys_stk4155_p1.regression.ordinary_least_squares import OLS
from fys_stk4155_p1.regression.ridge import Ridge
from fys_stk4155_p1.resampling.cross_validation import (
    kfold_mse_degree_sweep,
    kfold_mse_lasso_grid,
    kfold_mse_ridge_grid,
)

## Setup: data, design matrix, scaling, centering

No intercept column (`intercept=False`): every column of the design matrix is a
genuine polynomial term, `x^1 ... x^degree`, standardized on the training split. The
target is centered by its training-split mean; since standardized features have zero
mean, that mean is exactly the intercept the model would otherwise have fit — it's
just recovered separately (`y_train.mean()`) instead of as a `Lasso` coefficient.

In [ ]:
def standardize(X_train, X_test):
    """Standardize every column (fit on train only); no intercept column to skip."""
    scaler = StandardScaler()
    return scaler.fit_transform(X_train), scaler.transform(X_test)


x, y = generate_runge_data(n=200, noise_std=0.1, seed=42)
X_full = univariate_polynomial_design_matrix(x=x, degree=8, intercept=False)

X_train, X_test, y_train, y_test = train_test_split(X_full, y, test_size=0.2, random_state=42)
X_train, X_test = standardize(X_train, X_test)

y_mean = y_train.mean()
y_train_c = y_train - y_mean

We can plot the drawn data points on top of the Runge function as follows:

In [ ]:
x_plot = np.linspace(-1, 1, 500)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_plot, runge_function(x_plot), color="black", lw=1.5, label="Runge's function")
ax.scatter(x, y, s=15, alpha=0.6, label=r"data, $\sigma = 0.1$")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Runge function: noisy samples vs. ground truth")
ax.legend()
fig.tight_layout()
plt.show()

## Cross-validation

### OLS as a function of polynomial degree

We search degrees 0 to 15 using the 80/20 train/validation split we created earlier.

In [ ]:
degrees_ols = range(0, 16)
k = 5

cv_ols_result = kfold_mse_degree_sweep(x, y, degrees_ols, OLS, k=k, seed=42)

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(
    cv_ols_result["degrees"],
    cv_ols_result["mse_train_mean"],
    yerr=cv_ols_result["mse_train_std"],
    fmt="o-",
    markersize=4,
    color="tab:gray",
    capsize=2,
    label="train MSE (in-fold)",
)
ax.errorbar(
    cv_ols_result["degrees"],
    cv_ols_result["mse_mean"],
    yerr=cv_ols_result["mse_std"],
    fmt="s-",
    markersize=4,
    color="tab:blue",
    capsize=2,
    label="test MSE (held-out fold)",
)
best_degree = cv_ols_result["degrees"][np.argmin(cv_ols_result["mse_mean"])]
ax.axvline(
    best_degree,
    color="tab:blue",
    linestyle="--",
    linewidth=1,
    alpha=0.6,
    label=f"CV-selected degree = {best_degree}",
)

ax.set_yscale("log")
ax.set_xlabel("Polynomial degree")
ax.set_ylabel("MSE")
ax.set_title(f"OLS: {k}-fold CV train/test error vs. polynomial degree")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

### Ridge and Lasso as a function of Polynomial Degree and $\lambda$



In [ ]:
degrees_grid = range(0, 16)
lambdas_grid = np.logspace(-6, 1, 8 * 2)
k = 5

cv_ridge_result = kfold_mse_ridge_grid(x, y, degrees_grid, lambdas_grid, k=k, seed=42)

best_i, best_j = np.unravel_index(
    np.argmin(cv_ridge_result["mse_mean"]), cv_ridge_result["mse_mean"].shape
)
best_degree_ridge = cv_ridge_result["degrees"][best_i]
best_lambda_ridge = cv_ridge_result["lambdas"][best_j]

fig, ax = plt.subplots(figsize=(7, 4))
mesh = ax.pcolormesh(
    cv_ridge_result["degrees"],
    cv_ridge_result["lambdas"],
    cv_ridge_result["mse_mean"].T,
    norm=LogNorm(),
    shading="nearest",
    cmap="viridis_r",
)
ax.scatter(
    best_degree_ridge,
    best_lambda_ridge,
    marker="*",
    s=150,
    color="red",
    edgecolor="black",
    linewidth=0.5,
    label=f"best: degree={best_degree_ridge}, $\\lambda$={best_lambda_ridge:.1e}",
)
ax.set_yscale("log")
ax.set_xlabel("Polynomial degree")
ax.set_ylabel(r"$\lambda$")
ax.set_title(f"Ridge: {k}-fold CV test MSE over degree $\\times$ $\\lambda$")
ax.legend(frameon=False, loc="upper left", fontsize=8)
fig.colorbar(mesh, ax=ax, label="mean CV MSE")
fig.tight_layout()
plt.show()

In [ ]:
cv_lasso_result = kfold_mse_lasso_grid(x, y, degrees_grid, lambdas_grid, k=k, seed=42)

best_i, best_j = np.unravel_index(
    np.argmin(cv_lasso_result["mse_mean"]), cv_lasso_result["mse_mean"].shape
)
best_degree_lasso = cv_lasso_result["degrees"][best_i]
best_lambda_lasso = cv_lasso_result["lambdas"][best_j]

fig, ax = plt.subplots(figsize=(7, 4))
mesh = ax.pcolormesh(
    cv_lasso_result["degrees"],
    cv_lasso_result["lambdas"],
    cv_lasso_result["mse_mean"].T,
    norm=LogNorm(),
    shading="nearest",
    cmap="viridis_r",
)
ax.scatter(
    best_degree_lasso,
    best_lambda_lasso,
    marker="*",
    s=150,
    color="red",
    edgecolor="black",
    linewidth=0.5,
    label=f"best: degree={best_degree_lasso}, $\\lambda$={best_lambda_lasso:.1e}",
)
ax.set_yscale("log")
ax.set_xlabel("Polynomial degree")
ax.set_ylabel(r"$\lambda$")
ax.set_title(f"Lasso: {k}-fold CV test MSE over degree $\\times$ $\\lambda$")
ax.legend(frameon=False, loc="upper left", fontsize=8)
fig.colorbar(mesh, ax=ax, label="mean CV MSE")
fig.tight_layout()
plt.show()

### Final model comparison

The CV sweeps above already give a per-fold test MSE at every (degree, $\lambda$); the
minimum of each sweep is a legitimate CV estimate of that configuration's test error, and
comparing those three minima is already a valid way to rank OLS vs. Ridge vs. Lasso.

To get one final, head-to-head number per method (rather than an average over folds), we
take each method's CV-selected (degree, $\lambda$), refit **once** on a single 80/20
train/test split of the full data (using all of the training rows at once, not just
$(k-1)/k$ of them as within a fold), and report the held-out test MSE. Note this test
split isn't held out from the CV search itself (the sweeps above used the full dataset via
`KFold`), so this number is a refinement of the CV estimate for the final comparison, not
an independent generalization check.

In [ ]:
def final_test_mse(degree, model_factory):
    """Refit `model_factory()` at `degree` on one 80/20 train/test split and
    return the held-out test MSE. Scaling matches the CV pipelines above:
    intercept column passed through unscaled, `x^1..x^degree` standardized on
    the training rows only.
    """
    X_d = univariate_polynomial_design_matrix(x=x, degree=int(degree), intercept=True)
    X_tr, X_te, y_tr, y_te = train_test_split(X_d, y, test_size=0.2, random_state=42)
    if degree > 0:
        scaler = StandardScaler()
        X_tr[:, 1:] = scaler.fit_transform(X_tr[:, 1:])
        X_te[:, 1:] = scaler.transform(X_te[:, 1:])
    model = model_factory().fit(X_tr, y_tr)
    return mean_squared_error(y_te, model.predict(X_te))


final_results = {
    f"OLS (degree={best_degree})": final_test_mse(best_degree, OLS),
    f"Ridge (degree={best_degree_ridge}, $\\lambda$={best_lambda_ridge:.1e})": final_test_mse(
        best_degree_ridge, lambda: Ridge(lam=best_lambda_ridge, fit_intercept_column=True)
    ),
    f"Lasso (degree={best_degree_lasso}, $\\lambda$={best_lambda_lasso:.1e})": final_test_mse(
        best_degree_lasso,
        lambda: Lasso(
            learning_rate=0.05,
            lam=best_lambda_lasso,
            max_iter=3000,
            optimizer="adam",
            fit_intercept_column=True,
        ),
    ),
}

for name, mse in final_results.items():
    print(f"{name}: test MSE = {mse:.5f}")

best_model_name = min(final_results, key=final_results.get)
print(f"\nBest model overall: {best_model_name}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(final_results.keys(), final_results.values(), color=["tab:blue", "tab:orange", "tab:green"])
ax.set_ylabel("Held-out test MSE")
ax.set_title("Final model comparison")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
plt.show()